In [1]:
from summarization_dataset import BaseAbstractDataset, AdvancedAbstractDataset
from summarization_pipeline import SummarizationPipeline
from config import model_name_list, prompt_list, train_df, test_df, huggingface_token, openai_token
from util import read_csv, save_dataframe
import evaluation as ev

2025-03-07 14:27:27.617287: E external/local_xla/xla/stream_executor/cuda/cuda_fft.cc:485] Unable to register cuFFT factory: Attempting to register factory for plugin cuFFT when one has already been registered
2025-03-07 14:27:27.637877: E external/local_xla/xla/stream_executor/cuda/cuda_dnn.cc:8454] Unable to register cuDNN factory: Attempting to register factory for plugin cuDNN when one has already been registered
2025-03-07 14:27:27.644158: E external/local_xla/xla/stream_executor/cuda/cuda_blas.cc:1452] Unable to register cuBLAS factory: Attempting to register factory for plugin cuBLAS when one has already been registered
2025-03-07 14:27:27.659517: I tensorflow/core/platform/cpu_feature_guard.cc:210] This TensorFlow binary is optimized to use available CPU instructions in performance-critical operations.
To enable the following instructions: AVX2 FMA, in other operations, rebuild TensorFlow with the appropriate compiler flags.
2025-03-07 14:27:32.191010: W tensorflow/compiler/tf2

In [2]:
evaluate = ev.Evaluation()

/nfs/home/babaeih/.local/lib/python3.9/site-packages/spacy/util.py:910: UserWarning: [W095] Model 'en_core_web_sm' (3.5.0) was trained with spaCy v3.5.0 and may not be 100% compatible with the current version (3.8.3). If you see errors or degraded performance, download a newer compatible model or retrain your custom model with the current spaCy version. For more details and available updates, run: python -m spacy validate
  warnings.warn(warn_msg)


AttributeError: 'DownloadConfig' object has no attribute 'use_auth_token'

In [ ]:
# metric = evaluate.load("accuracy")
# !pip install huggingface_hub -U

In [3]:
bleu = evaluate.load("bleu")

NameError: name 'evaluate' is not defined

In [1]:
# !pip install evaluate

In [8]:
# os.environ['HUGGINGFACE_ACCESS_TOKEN']

In [2]:
import evaluate
# from huggingface_hub import login
# import os

# login(token=os.environ['HUGGINGFACE_ACCESS_TOKEN'])

bleu = evaluate.load("bleu")
rouge = evaluate.load("rouge")


2025-03-07 14:30:18.840919: E external/local_xla/xla/stream_executor/cuda/cuda_fft.cc:485] Unable to register cuFFT factory: Attempting to register factory for plugin cuFFT when one has already been registered
2025-03-07 14:30:18.861247: E external/local_xla/xla/stream_executor/cuda/cuda_dnn.cc:8454] Unable to register cuDNN factory: Attempting to register factory for plugin cuDNN when one has already been registered
2025-03-07 14:30:18.867538: E external/local_xla/xla/stream_executor/cuda/cuda_blas.cc:1452] Unable to register cuBLAS factory: Attempting to register factory for plugin cuBLAS when one has already been registered
2025-03-07 14:30:18.883004: I tensorflow/core/platform/cpu_feature_guard.cc:210] This TensorFlow binary is optimized to use available CPU instructions in performance-critical operations.
To enable the following instructions: AVX2 FMA, in other operations, rebuild TensorFlow with the appropriate compiler flags.
2025-03-07 14:30:23.490261: W tensorflow/compiler/tf2

In [6]:
bleu = evaluate.load("bleu", use_auth_token= os.environ['HUGGINGFACE_ACCESS_TOKEN'])

AttributeError: 'DownloadConfig' object has no attribute 'use_auth_token'

In [ ]:
import spacy
import evaluate
from bert_score import BERTScorer
import pandas as pd
import os

class Evaluator:
  def __init__(self, bert_model_name='allenai/scibert_scivocab_uncased', model_name='en_core_web_sm'):
    self.bert_scorer = BERTScorer(model_type=bert_model_name)
    self.nlp = spacy.load(model_name) 
    self.bleu = evaluate.load("bleu", use_auth_token=os.environ['HUGGINGFACE_ACCESS_TOKEN'])
    self.rouge = evaluate.load("rouge", use_auth_token=os.environ['HUGGINGFACE_ACCESS_TOKEN'])
  

In [ ]:
for model_name in model_name_list:
    print(f"Working on {model_name}!")
    dataset = read_csv(train_df).sample(10)
    for prompt_name, prompt in prompt_list.items():
        print(f"\tWork in progress for {prompt_name} based prompting...")
        pipeline = SummarizationPipeline(model_name=model_name,
                                        load_type="qlora",
                                        batch_size=4,
                                        token=huggingface_token
                                        )
        if prompt_name != "dsp_prompt":
            output = pipeline.run(dataset=dataset,
                        dataset_class=BaseAbstractDataset,
                        prompt=prompt)
        else:
            output = pipeline.run(dataset=dataset,
                        dataset_class=AdvancedAbstractDataset,
                        prompt=prompt)
        dataset[prompt_name] = output
        dataset = evaluate.evaluate_summary(df=dataset, prompt_name=prompt_name)
    file_name = f"{model_name}-evaluation.csv"
    save_dataframe(df=dataset, directory="results", filename=file_name)
